<a href="https://colab.research.google.com/github/agrivera89/Internal-linking-analysis/blob/main/vectordb_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vector Databases

## Setup

In [ ]:
!pip install -qq chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.2/65.2 kB 4.4 MB/s eta 0:0

In [ ]:
!pip install -q sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 60.8 MB/s eta 0:00:00


In [ ]:
import torch
torch.__version__

'2.6.0+cu124'

In [ ]:
torch.cuda.is_available()

False

## Repaso: Embedding

Representación vectorial de un item, que puede ser, por ejemplo: texto, imágenes, audio o video.

Se llama vector a un segmento de recta en el espacio que parte de un punto hacia otro, es decir, que tiene dirección y sentido.

## Ejercicio 0

Vamos a usar datos ficticios de comentarios sobre productos de indumentaria y hacer algunas queries.

Para esto, haremos lo siguiente:

1. Crear una base de datos vectorial (ChromaDB)
2. Crear colección de datos e indexar la data
3. Realizar una búsqueda semántica: escribir una query que devuelva una lista de documentos en base a similaridad
4. Insertar nueva data y correr la misma query del punto 3

### 1. ChromaDB

ChromaDB es una base de datos vectorial open-source ([Github](https://github.com/chroma-core/chroma)).  
Fue diseñada para operar principalmente sobre texto y servir como base de conocimiento para LLMs como GPT-3.

In [ ]:
import chromadb

# Creamos la base de datos in-memory para este ejercicio.
client = chromadb.Client()

### 2. Collections

Una colección es el contenedor en el que almacenaremos y organizaremos los datos junto a sus embeddings. Es similar a una tabla en una base de datos relacional, o a una colección en MongoDB.

Soporta operaciones **CRUD** (Create, Read, Update, Delete)

In [ ]:
client.list_collections()

[]

In [ ]:
# Otras funciones son `get_collection`, `get_or_create_collection` y `delete_collection`.
# client.delete_collection(name="Comments")

collection = client.create_collection(
    name="Comments",
#     # metadata={"hnsw:space": "cosine"} # Opcional
#     # embedding_function=emb_fn # default: Sentence Transformers all-MiniLM-L6-v2
)

In [ ]:
client.list_collections()

['Comments']

Cuando agregamos documentos a la colección, Chroma se encarga automáticamente de obtener los embeddings y del indexado.

Por default, ChromaDB usa el modelo **`all-MiniLM-L6-v2`** [(disponible en HuggingFace)](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) y distancia L2 (Euclídea) como métrica.

**Nota:** HuggingFace es como el Github de los modelos de machine learning.

In [ ]:
# Agregamos docs a la collection.
comments = [
    "Me encantan las zapatillas que compré, son muy cómodas.",
    "Las zapatillas estaban un poco apretadas, recomendaría comprar una talla más.",
    "El bolso gratis fue una sorpresa agradable, gracias!",
    "Compré tres productos y estoy satisfecho con la calidad de todos ellos.",
]

## Chroma se encarga automáticamente del embedding y el indexado
collection.add(
    documents=comments,
    metadatas=[{"source": "google-form"}, {"source": "google-form"}, {"source": "instagram"}, {"source": "instagram"}],  # Se puede usar la metadata para filtrar
    ids=["doc1", "doc2", "doc3", "doc4"],  # Id único por cada doc
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 35.5MiB/s]


In [ ]:
# Podemos verificar que los documentos fueron agregados
# contando la cantidad de documentos en la colección.
collection.count()

4

In [ ]:
# Usar `get` sin especificar el id funciona como un SELECT * en SQL.
collection.get()

{'ids': ['doc1', 'doc2', 'doc3', 'doc4'],
 'embeddings': None,
 'documents': ['Me encantan las zapatillas que compré, son muy cómodas.',
  'Las zapatillas estaban un poco apretadas, recomendaría comprar una talla más.',
  'El bolso gratis fue una sorpresa agradable, gracias!',
  'Compré tres productos y estoy satisfecho con la calidad de todos ellos.'],
 'uris': None,
 'data': None,
 'metadatas': [{'source': 'google-form'},
  {'source': 'google-form'},
  {'source': 'instagram'},
  {'source': 'instagram'}],
 'included': [<IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

In [ ]:
# También podemos usar el id para obtener un documento específico.
collection.get('doc2')

{'ids': ['doc2'],
 'embeddings': None,
 'documents': ['Las zapatillas estaban un poco apretadas, recomendaría comprar una talla más.'],
 'uris': None,
 'data': None,
 'metadatas': [{'source': 'google-form'}],
 'included': [<IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

### 3. Búsqueda semántica

In [ ]:
# Buscamos los 3 resultados más similares a la query.
user_question = "No me gustó el bolso gratis"

results = collection.query(
    query_texts=[user_question],
    n_results=3,
    #where={"metadata_field": "google-form"}, # Opcional
    # where_document={"$contains": "bolso"}  # Opcional
)

results

{'ids': [['doc3', 'doc4', 'doc1']],
 'embeddings': None,
 'documents': [['El bolso gratis fue una sorpresa agradable, gracias!',
   'Compré tres productos y estoy satisfecho con la calidad de todos ellos.',
   'Me encantan las zapatillas que compré, son muy cómodas.']],
 'uris': None,
 'data': None,
 'metadatas': [[{'source': 'instagram'},
   {'source': 'instagram'},
   {'source': 'google-form'}]],
 'distances': [[0.4885711967945099, 1.040198564529419, 1.1001567840576172]],
 'included': [<IncludeEnum.distances: 'distances'>,
  <IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

In [ ]:
# Podemos hacer una consulta sin búsqueda semántica, usando `get`.
collection.get(where_document={"$contains": "bolso"})

{'ids': ['doc3'],
 'embeddings': None,
 'documents': ['El bolso gratis fue una sorpresa agradable, gracias!'],
 'uris': None,
 'data': None,
 'metadatas': [{'source': 'instagram'}],
 'included': [<IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

### 4. Agregar nuevos comentarios a la colección
Y volvemos a ejecutar la misma búsqueda semántica

In [ ]:
new_instagram_comments = [
    "Las zapatillas son de excelente calidad, se ajustan perfectamente y son muy cómodas para correr.",
    "La cartera que recibí con mi compra parece barata y no es muy duradera.",
    "He comprado varias veces aquí y siempre quedo satisfecho con los productos.",
    "La cartera de regalo fue una decepción, no coincidía con las imágenes en línea.",
    "Me encanta la variedad de productos, siempre encuentro lo que necesito y la entrega es rápida."
]

In [ ]:
def prepare_collection_args(comments, source, current_max_id):
    metadatas = [{"source": source} for _ in comments]
    new_ids = [f"doc{current_max_id + i + 1}" for i in range(len(comments))]

    return {
        "documents": comments,
        "metadatas": metadatas,
        "ids": new_ids
    }

In [ ]:
current_max_id = collection.count()

instagram_args = prepare_collection_args(new_instagram_comments, "instagram", current_max_id)

collection.add(**instagram_args)

In [ ]:
collection.get()

{'ids': ['doc1',
  'doc2',
  'doc3',
  'doc4',
  'doc5',
  'doc6',
  'doc7',
  'doc8',
  'doc9'],
 'embeddings': None,
 'documents': ['Me encantan las zapatillas que compré, son muy cómodas.',
  'Las zapatillas estaban un poco apretadas, recomendaría comprar una talla más.',
  'El bolso gratis fue una sorpresa agradable, gracias!',
  'Compré tres productos y estoy satisfecho con la calidad de todos ellos.',
  'Las zapatillas son de excelente calidad, se ajustan perfectamente y son muy cómodas para correr.',
  'La cartera que recibí con mi compra parece barata y no es muy duradera.',
  'He comprado varias veces aquí y siempre quedo satisfecho con los productos.',
  'La cartera de regalo fue una decepción, no coincidía con las imágenes en línea.',
  'Me encanta la variedad de productos, siempre encuentro lo que necesito y la entrega es rápida.'],
 'uris': None,
 'data': None,
 'metadatas': [{'source': 'google-form'},
  {'source': 'google-form'},
  {'source': 'instagram'},
  {'source': 'i

In [ ]:
user_question = "No me gustó el bolso gratis"

results = collection.query(
    query_texts=[user_question],
    n_results=3,
    # where={"metadata_field": "valor_de_metadata"}, # Opcional
    # where_document={"$contains": "keyword"}  # Opcional
)

print("BÚSQUEDA SEMÁNTICA\n")
results

BÚSQUEDA SEMÁNTICA



{'ids': [['doc3', 'doc6', 'doc7']],
 'embeddings': None,
 'documents': [['El bolso gratis fue una sorpresa agradable, gracias!',
   'La cartera que recibí con mi compra parece barata y no es muy duradera.',
   'He comprado varias veces aquí y siempre quedo satisfecho con los productos.']],
 'uris': None,
 'data': None,
 'metadatas': [[{'source': 'instagram'},
   {'source': 'instagram'},
   {'source': 'instagram'}]],
 'distances': [[0.4885711967945099, 1.002102255821228, 1.0185049772262573]],
 'included': [<IncludeEnum.distances: 'distances'>,
  <IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

In [ ]:
print("BÚSQUEDA POR KEYWORDS\n")
# Usamos $or para filtrar por más de una keyword.
collection.get(where_document={"$or": [{"$contains": "bolso"}, {"$contains": "cartera"}]})

BÚSQUEDA POR KEYWORDS



{'ids': ['doc3', 'doc6', 'doc8'],
 'embeddings': None,
 'documents': ['El bolso gratis fue una sorpresa agradable, gracias!',
  'La cartera que recibí con mi compra parece barata y no es muy duradera.',
  'La cartera de regalo fue una decepción, no coincidía con las imágenes en línea.'],
 'uris': None,
 'data': None,
 'metadatas': [{'source': 'instagram'},
  {'source': 'instagram'},
  {'source': 'instagram'}],
 'included': [<IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

Como podemos ver, en la búsqueda semántica no me trajo todos los documentos que hagan mención de un bolso o cartera. De hecho, el primer resultado es un comentario positivo hacia el bolso gratis.

Lo que está pasando, es que el modelo de embeddings por default que está usando Chroma está entrenado sólo con texto en inglés.

En el próximo, vamos a cambiar tanto el modelo de embedding, para que sea multilenguaje, como la métrica de distancia.



Siguientes pasos:  

5. Descargar un modelo de embeddings multilenguaje
6. Crear una colección nueva con este modelo y usar el coseno como métrica de distancia
7. Ejecutar queries y evaluar resultados

### 5. Descargar Embedding multilenguaje

El modelo que vamos a usar es el `multilingual-e5-large` ([HuggingFace](https://huggingface.co/intfloat/multilingual-e5-small)). Chroma nos permite usar modelos de embeddings de texto desde HuggingFace con el método `SentenceTransformerEmbeddingFunction`.

Este modelo soporta 94 idiomas distintos, incluído el Español.

In [ ]:
from chromadb.utils import embedding_functions

sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="intfloat/multilingual-e5-large")
# Otra opción: hiiamsid/sentence_similarity_spanish_es

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

### 6. Crear colección nueva

In [ ]:
# client.delete_collection(name="comments_multilingual")

In [ ]:
collection = client.get_or_create_collection(
    "comments_multilingual",
    embedding_function=sentence_transformer_ef,
    metadata={"hnsw:space": "cosine"}
)

In [ ]:
client.list_collections()

['comments_multilingual', 'Comments']

In [ ]:
# Agregamos docs a la collection.
collection.add(
    documents=comments,
    metadatas=[{"source": "google-form"}, {"source": "google-form"}, {"source": "instagram"}, {"source": "instagram"}],  # Se puede usar la metadata para filtrar
    ids=["doc1", "doc2", "doc3", "doc4"],  # Id único por cada doc
)

collection.add(
    documents=new_instagram_comments,
    metadatas=[{"source": "instagram"}, {"source": "instagram"}, {"source": "instagram"}, {"source": "instagram"}, {"source": "instagram"}],
    ids=["doc5", "doc6", "doc7", "doc8", "doc9"],
)

### 7. Ejecutar búsqueda semántica con los nuevos embeddings

In [ ]:
user_question = "No me gustó el bolso gratis"

results = collection.query(
    query_texts=[user_question],
    n_results=3
)

results

{'ids': [['doc3', 'doc6', 'doc8']],
 'embeddings': None,
 'documents': [['El bolso gratis fue una sorpresa agradable, gracias!',
   'La cartera que recibí con mi compra parece barata y no es muy duradera.',
   'La cartera de regalo fue una decepción, no coincidía con las imágenes en línea.']],
 'uris': None,
 'data': None,
 'metadatas': [[{'source': 'instagram'},
   {'source': 'instagram'},
   {'source': 'instagram'}]],
 'distances': [[0.11355537176132202, 0.1464133858680725, 0.1541736125946045]],
 'included': [<IncludeEnum.distances: 'distances'>,
  <IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

Usando un embedding multilenguaje, pudimos capturar la similaridad entre bolso y cartera. Aún así, podemos ver que la mención explícita de "bolso gratis" sigue teniendo más peso que si el comentario es positivo o negativo.